In [ ]:
!pip install groq fastapi uvicorn python-dotenv

In [5]:
import os
import json
import time
import threading
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse

# ---------------------------------------------------------------------
# MEMORY -- the single source of truth for the whole chatbot.
# Every user message, assistant reply, and (later) tool call/result gets
# appended here. The chat UI only ever displays "user" and "assistant"
# turns that have text -- everything else stays invisible to the user.
# ---------------------------------------------------------------------
SYSTEM_MESSAGE = {"role": "system", "content": "You are a helpful assistant.Your name is EMilie"}
messages = [dict(SYSTEM_MESSAGE)]


def add_message(role, content, **extra):
    msg = {"role": role, "content": content, **extra}
    messages.append(msg)
    return msg


def visible_messages():
    """What the browser is allowed to see: text-bearing user/assistant turns."""
    return [
        {"role": m["role"], "content": m["content"]}
        for m in messages
        if m["role"] in ("user", "assistant") and m.get("content")
    ]


# ---------------------------------------------------------------------
# THE "BRAIN SLOT" -- every later cell in this notebook REPLACES this
# function. The FastAPI route below always calls whatever RESPOND
# currently points to, so re-running a cell instantly upgrades the
# live UI. No server restart required.
# ---------------------------------------------------------------------
def RESPOND():
    """Level 0: memory only. Nothing streams back -- there's no brain yet."""
    yield ""


# ---------------------------------------------------------------------
# The chat webpage itself -- plain HTML + JS, zero external dependencies.
# ---------------------------------------------------------------------
CHAT_HTML = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Tool Calling Demo Chat</title>
<style>
  * { box-sizing: border-box; }
  body {
    margin: 0; font-family: -apple-system, Segoe UI, Roboto, sans-serif;
    background: #0f1115; color: #e6e6e6; height: 100vh; display: flex; flex-direction: column;
  }
  header {
    padding: 14px 20px; border-bottom: 1px solid #23262e; font-weight: 600;
    display: flex; align-items: center; justify-content: space-between;
  }
  #clearBtn {
    padding: 6px 12px; border-radius: 8px; border: 1px solid #2a2e38; background: transparent;
    color: #8b90a0; font-size: 12px; font-weight: 500; cursor: pointer;
  }
  #clearBtn:hover { background: #1c1f27; color: #e6e6e6; border-color: #3a3f4b; }
  #chat { flex: 1; overflow-y: auto; padding: 20px; display: flex; flex-direction: column; gap: 12px; }
  .bubble { max-width: 70%; padding: 10px 14px; border-radius: 14px; line-height: 1.45; white-space: pre-wrap; }
  .user { align-self: flex-end; background: #2f6feb; color: white; border-bottom-right-radius: 4px; }
  .assistant { align-self: flex-start; background: #1c1f27; border: 1px solid #2a2e38; border-bottom-left-radius: 4px; }
  .assistant.pending { color: #8b90a0; font-style: italic; }
  .assistant code { background: #11131a; padding: 1px 5px; border-radius: 4px; font-size: 0.9em; }
  .assistant pre { background: #11131a; padding: 10px; border-radius: 8px; overflow-x: auto; }
  #inputRow { display: flex; gap: 10px; padding: 14px 20px; border-top: 1px solid #23262e; }
  #textInput { flex: 1; padding: 10px 14px; border-radius: 10px; border: 1px solid #2a2e38; background: #1c1f27; color: #e6e6e6; font-size: 14px; }
  #sendBtn { padding: 10px 18px; border-radius: 10px; border: none; background: #2f6feb; color: white; font-weight: 600; cursor: pointer; }
  #sendBtn:disabled { opacity: 0.5; cursor: not-allowed; }
</style>
</head>
<body>
  <header>
    <span>Tool Calling Demo Chat</span>
    <button id="clearBtn" title="Wipe backend memory and start over">Clear memory</button>
  </header>
  <div id="chat"></div>
  <div id="inputRow">
    <input id="textInput" placeholder="Type a message and hit Enter..." autocomplete="off">
    <button id="sendBtn">Send</button>
  </div>
<script>
const chatEl = document.getElementById('chat');
const inputEl = document.getElementById('textInput');
const sendBtn = document.getElementById('sendBtn');
const clearBtn = document.getElementById('clearBtn');

function renderMD(text) {
  const esc = text.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');
  return esc
    .replace(/```([\\s\\S]*?)```/g, '<pre><code>$1</code></pre>')
    .replace(/`([^`]+)`/g, '<code>$1</code>')
    .replace(/\\*\\*([^*]+)\\*\\*/g, '<strong>$1</strong>')
    .replace(/\\*([^*]+)\\*/g, '<em>$1</em>')
    .replace(/\\n/g, '<br>');
}

function addBubble(role, text) {
  const div = document.createElement('div');
  div.className = 'bubble ' + role;
  div.innerHTML = renderMD(text);
  chatEl.appendChild(div);
  chatEl.scrollTop = chatEl.scrollHeight;
  return div;
}

async function loadHistory() {
  const res = await fetch('/api/messages');
  const msgs = await res.json();
  chatEl.innerHTML = '';
  msgs.forEach(m => addBubble(m.role, m.content));
}

async function sendMessage() {
  const text = inputEl.value.trim();
  if (!text) return;
  inputEl.value = '';
  sendBtn.disabled = true;

  addBubble('user', text);
  const pending = addBubble('assistant', '...');
  pending.classList.add('pending');

  const res = await fetch('/api/stream', {
    method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({text})
  });

  const reader = res.body.getReader();
  const decoder = new TextDecoder();
  let full = '';
  while (true) {
    const {done, value} = await reader.read();
    if (done) break;
    full += decoder.decode(value, {stream: true});
    pending.classList.remove('pending');
    pending.innerHTML = renderMD(full || '...');
    chatEl.scrollTop = chatEl.scrollHeight;
  }

  // Resync fully with backend memory (source of truth) once streaming ends.
  await loadHistory();
  sendBtn.disabled = false;
  inputEl.focus();
}

async function clearMemory() {
  clearBtn.disabled = true;
  await fetch('/api/clear', { method: 'POST' });
  await loadHistory();
  clearBtn.disabled = false;
  inputEl.focus();
}

sendBtn.addEventListener('click', sendMessage);
inputEl.addEventListener('keydown', e => { if (e.key === 'Enter') sendMessage(); });
clearBtn.addEventListener('click', clearMemory);

loadHistory();
</script>
</body>
</html>
"""

# ---------------------------------------------------------------------
# FastAPI server -- started ONCE, in a background thread.
# ---------------------------------------------------------------------
app = FastAPI()


@app.get("/")
async def index():
    return HTMLResponse(CHAT_HTML)


@app.get("/api/messages")
async def get_messages():
    return JSONResponse(visible_messages())


@app.post("/api/stream")
async def stream(request: Request):
    data = await request.json()
    user_text = data.get("text", "")
    add_message("user", user_text)

    def gen():
        # RESPOND is a plain sync generator -- Starlette runs it in a
        # threadpool under the hood, so it won't block the event loop.
        yield from RESPOND()

    return StreamingResponse(gen(), media_type="text/plain")


@app.post("/api/clear")
async def clear():
    """Wipe memory back down to just the system prompt -- start from scratch
    without restarting the server or losing whichever RESPOND level you're on."""
    messages.clear()
    messages.append(dict(SYSTEM_MESSAGE))
    return JSONResponse({"ok": True})


PORT = 8765


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")


if not globals().get("_SERVER_STARTED"):
    threading.Thread(target=run_server, daemon=True).start()
    _SERVER_STARTED = True
    time.sleep(1.5)  # give uvicorn a moment to actually bind the port

print(f"Chat UI running at http://127.0.0.1:{PORT}")
print("Open that URL in a browser tab and keep it open -- later cells upgrade it live.")


Chat UI running at http://127.0.0.1:8765
Open that URL in a browser tab and keep it open -- later cells upgrade it live.


In [4]:
from dotenv import load_dotenv
from groq import Groq

load_dotenv()  # reads GROQ_API_KEY (and friends) from a .env file in this folder

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"


def RESPOND():
    """Level 1: the brain is connected. Streams a real LLM reply -- no tools yet."""
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": m["role"], "content": m["content"]} for m in messages],
        stream=True,
    )
    full_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if delta:
            full_text += delta
            yield delta
    add_message("assistant", full_text)


print("Brain connected. Go to the browser tab and send a message -- it replies for real now.")


Brain connected. Go to the browser tab and send a message -- it replies for real now.


In [8]:
def get_weather(city: str) -> str:
    """Pretend weather API -- swap this for a real API call if you like."""
    fake_db = {
        "pune": "28°C, partly cloudy",
        "mumbai": "31°C, humid",
        "delhi": "24°C, clear skies",
    }
    return fake_db.get(city.lower(), f"No weather data for {city}")


def calculator(expression: str) -> str:
    """Evaluates a basic arithmetic expression, e.g. '12 * (4 + 3)'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


# --- These two are REAL, not pretend: they touch the actual filesystem in
# the notebook's own working directory. That's what makes the final demo
# feel realistic instead of toy-ish.
WORKDIR = os.getcwd()


def _safe_path(path: str) -> str:
    """Keep every file tool scoped to this notebook's own directory --
    stops the model from being tricked into touching files elsewhere."""
    full = os.path.abspath(os.path.join(WORKDIR, path))
    if full != WORKDIR and not full.startswith(WORKDIR + os.sep):
        raise ValueError(f"'{path}' escapes the working directory -- not allowed")
    return full


def read_file(path: str) -> str:
    """Read and return the contents of a text file in the current directory."""
    try:
        with open(_safe_path(path), "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        return f"Error reading {path}: {e}"


def write_file(path: str, content: str) -> str:
    """Create (or overwrite) a text file in the current directory with the given content."""
    try:
        full = _safe_path(path)
        os.makedirs(os.path.dirname(full) or WORKDIR, exist_ok=True)
        with open(full, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Wrote {len(content)} characters to {path}"
    except Exception as e:
        return f"Error writing {path}: {e}"


print("Functions defined (including real file read/write). RESPOND is unchanged -- go try asking for the file operations in the browser, it still fails.")


Functions defined (including real file read/write). RESPOND is unchanged -- go try asking for the file operations in the browser, it still fails.


In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. Pune"}
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "e.g. '12 * (4 + 3)'"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a text file from the current working directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Relative file path, e.g. 'notes.txt'"}
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Create or overwrite a text file in the current working directory with the given content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Relative file path, e.g. 'notes.txt'"},
                    "content": {"type": "string", "description": "The text to write into the file"},
                },
                "required": ["path", "content"],
            },
        },
    },
]


def _trimmed(m):
    """Only forward the fields the Groq API accepts."""
    keys = ("role", "content", "tool_calls", "tool_call_id", "name")
    return {k: m[k] for k in keys if k in m}


def RESPOND():
    """Level 2: tools registered. The model can now ASK to call one -- but
    we deliberately do NOT execute it yet, just surface the raw request."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[_trimmed(m) for m in messages],
        tools=tools,
    )  # non-streaming on purpose, so we can inspect the whole tool_calls object at once
    msg = response.choices[0].message

    if msg.tool_calls:
        tc = msg.tool_calls[0]
        text = (
            f"🔧 Model wants to call **{tc.function.name}** "
            f"with args `{tc.function.arguments}` — (not executed yet, see next cells)"
        )
    else:
        text = msg.content

    add_message("assistant", text)
    yield text


print("Tools registered (weather, calculator, read_file, write_file). Go ask for the weather again -- watch it ASK instead of ANSWER.")


Tools registered (weather, calculator, read_file, write_file). Go ask for the weather again -- watch it ASK instead of ANSWER.


## 5️⃣ Extract the function name and arguments

The arguments arrive as a JSON **string**, not a Python dict — we must parse them before we can call the real Python function with them. This cell runs a fresh, self-contained request (independent of whatever's in the live chat) just to inspect that raw object clearly.


In [17]:
demo_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "use writefile and create test.py with hello world "},
]

demo_response = client.chat.completions.create(
    model=MODEL,
    messages=demo_messages,
    tools=tools,
)
# print(demo_response)
tool_call = demo_response.choices[0].message.tool_calls[0]
print(tool_call)

function_name = tool_call.function.name
function_args = json.loads(tool_call.function.arguments)  # string -> dict

print("Model wants to call:", function_name)
print("With arguments:", function_args)


ChatCompletionMessageToolCall(id='fc_41cf135a-1514-4229-80b8-d0083157b695', function=Function(arguments='{"content":"print(\\"Hello, World!\\")","path":"test.py"}', name='write_file'), type='function')
Model wants to call: write_file
With arguments: {'content': 'print("Hello, World!")', 'path': 'test.py'}


## 6️⃣ Execute the tool, send the result back, and complete the loop

The final version of `RESPOND`:

1. Asks the model what it wants to do
2. If it requests a tool call: look up the real Python function, run it with the extracted arguments, append the result to memory as a `role: "tool"` message linked via `tool_call_id`
3. Loop back to step 1 — now the model has real data and can write a normal, natural-language answer
4. If it returns plain text instead: stream that out and stop

Go to the browser and try it for real:

- *"What's the weather in Pune, and what's 15 * 12?"*
- *"Write a file called notes.txt with a short haiku about tools"* — this actually creates the file in this notebook's folder
- *"Now read notes.txt back to me"* — this actually reads it off disk

Check your file explorer afterward — `notes.txt` will really be sitting there. That's the whole point: this isn't a simulation of tool calling, it's the real thing.


In [18]:
available_functions = {
    "get_weather": get_weather,
    "calculator": calculator,
    "read_file": read_file,
    "write_file": write_file,
}


def RESPOND():
    """Level 3: the full tool-calling loop. Keeps going until the model
    returns a plain-text answer -- this is what a production chatbot does."""
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[_trimmed(m) for m in messages],
            tools=tools,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            add_message("assistant", msg.content)
            yield msg.content
            return

        # Record the request itself (content=None -> invisible in the UI,
        # but still part of memory so the model sees its own past turn).
        add_message(
            "assistant",
            None,
            tool_calls=[
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ],
        )

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            result = available_functions[name](**args)  # <-- the real function actually runs
            add_message("tool", str(result), tool_call_id=tc.id, name=name)

        # loop again -- the model now has real data and can answer for real


print("Full loop wired up. Go ask for the weather, a calculation, or 'write a file called notes.txt with a haiku about tools' -- then 'read notes.txt back to me'.")


Full loop wired up. Go ask for the weather, a calculation, or 'write a file called notes.txt with a haiku about tools' -- then 'read notes.txt back to me'.


## 🧠 Peek at memory vs. what the browser shows

Print `messages` below — this is everything the backend tracked across the whole conversation, including the invisible `tool_calls` request and the `role: "tool"` result. The browser only ever rendered the `user` bubbles and the final `assistant` text bubbles (via `/api/messages` → `visible_messages()`). Everything else was working quietly behind the scenes — that gap between the two is exactly what "tool calling" is.


In [ ]:
for m in messages:
    shown = {k: v for k, v in m.items() if k != "role"}
    print(f"{m.get('role'):9s} -> {shown}")


In [3]:
!pip install groq python-dotenv

In [4]:
import os 
import json 
from dotenv import load_dotenv
load_dotenv()
from groq import Groq 

model="openai/gpt-oss-120b"
client=Groq(api_key=os.environ.get("Groq_API_KEY"))
memory_list = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    }
]

In [5]:
def call_model(messages,tools=None):
    tool_response=None
    response=client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
        
    )
    if tools:
        tool_response=response.choices[0].message.tool_calls
    return response.choices[0].message.content,tool_response

In [6]:

def append_message(role, content, **extra):
    msg = {
        "role": role,
        "content": content,
        **extra
    }

    memory_list.append(msg)
    return memory_list

In [12]:
message=append_message("user","use my function write_file as tool and create a file called test.py with hello world")

response,tool_response=call_model(message,tools)
print(tool_response)
print(response)

[ChatCompletionMessageToolCall(id='fc_e348fbda-d69d-442b-b2f9-6fee7602b73b', function=Function(arguments='{"content":"hello world","path":"test.py"}', name='write_file'), type='function')]
None


In [13]:
print("Tool response:", tool_response)
args=json.loads(tool_response[0].function.arguments)
print(args)

result=write_file(args["path"],args["content"])

message=append_message("tool",result,tool_call_id=tool_response[0].id,name=tool_response[0].function.name)

response=call_model(message,tools)
print(response)



Tool response: [ChatCompletionMessageToolCall(id='fc_e348fbda-d69d-442b-b2f9-6fee7602b73b', function=Function(arguments='{"content":"hello world","path":"test.py"}', name='write_file'), type='function')]
{'content': 'hello world', 'path': 'test.py'}


BadRequestError: Error code: 400 - {'error': {'message': "'messages.3' : for 'role:tool' the following must be satisfied[('messages.3.content' : one of the following must be satisfied[('messages.3.content' : Value is not nullable) OR ('messages.3.content' : Value is not nullable)])]", 'type': 'invalid_request_error'}}

In [9]:
def write_file(path:str,content:str):
    try:
        with open(path,"w",encoding=
                  "utf-8") as f:
            f.write(content)
    except Exception as e:
        return f"Error writing {path}: {e}"
    

In [51]:
write_file("test.py","hello world")

In [11]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write the provided content to a file at the specified path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path where the file should be created or overwritten."
                    },
                    "content": {
                        "type": "string",
                        "description": "The text content to write into the file."
                    }
                },
                "required": ["path", "content"]
            }
        }
    }
]